# Use the Trained OADM + ESM-2 8M Antibody Model (Inference Notebook)

This notebook lets you **load the trained baseline model and generate paired antibody
sequences** — no training, no data pipeline. It is self-contained: the install cell below
stands in for a `requirements.txt`, and the load + `generate`/`inpaint` cells stand in for a
`generate.py`, so you don't need any other code files.

**What you need (the release bundle):** the `baseline_release` folder *or* `baseline_release.zip`.
It must contain:
- `model.safetensors` (or `pytorch_model.bin`) and `config.json` — the trained weights + architecture
- `tokenizer_config.json`, `vocab.txt`, `special_tokens_map.json` — the tokenizer (carries the
  added `|` token and resized vocab; **required**, the model breaks with a stock ESM-2 tokenizer)
- `length_pairs.json` — empirical (heavy_len, light_len) distribution for sampling lengths
- `config.json.bak` — the training run's settings (used here to pick faithful generation defaults)

**What you do NOT need:** the training corpus CSV, the `tok_*.pt` caches, `ckpt_latest.pt`,
`metrics.csv`, or the data pipeline. Those are only for retraining/extending the run.

**Model background:** masked discrete-diffusion (OADM) over `<heavy>|<light>` sequences, ESM-2 8M
architecture. Reverse generation unmasks one position at a time (most-confident-first) and samples
each token at a temperature. See the project README for full details.


## 1. Install dependencies

This replaces a `requirements.txt`.

In [ ]:
# === Install dependencies (Colab) ============================================
# Pinned to the versions the model was trained/saved under (from config.json:
# "transformers_version": "5.12.0"). torch is pinned loosely -- it rarely affects
# ESM inference, and the exact training build (2.x +cuXXX) depends on the runtime.
%pip -q install "transformers==5.12.0" "torch>=2.1" pandas numpy
# Optional, only for paper-style naturalness scoring (heavy; see last section):
# %pip -q install ablang2 pigen

import torch
from transformers import AutoTokenizer, EsmForMaskedLM
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("deps installed")

In [ ]:
# === Imports & device =========================================================
import os, json, random, zipfile

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
def set_seed(s=0):
    random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("device:", DEVICE)

## 2. Point to the model

Edit `MODEL_DIR` to the unzipped `baseline_release` folder. If you only have the zip, set
`ZIP_PATH` and the cell will extract it for you.

**On Google Colab**, mount Drive first and use the Drive path, e.g.
`MODEL_DIR = "/content/drive/MyDrive/.../baseline_release"`:
```python
# from google.colab import drive; drive.mount('/content/drive')
```


In [ ]:
# === Locate / unzip the release ==============================================
MODEL_DIR = "baseline_release"        # <-- EDIT: the unzipped release folder
ZIP_PATH  = "baseline_release.zip"    # <-- EDIT: used only if MODEL_DIR doesn't exist yet

if not os.path.isdir(MODEL_DIR):
    assert os.path.exists(ZIP_PATH), (
        f"Found neither folder '{MODEL_DIR}' nor zip '{ZIP_PATH}'. "
        "Point MODEL_DIR at the release folder.")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(MODEL_DIR)
    print(f"extracted {ZIP_PATH} -> {MODEL_DIR}/")

# sanity: required files present
has_weights = any(os.path.exists(os.path.join(MODEL_DIR, w))
                  for w in ("model.safetensors", "pytorch_model.bin"))
need = ["config.json", "length_pairs.json"]
missing = [f for f in need if not os.path.exists(os.path.join(MODEL_DIR, f))]
assert has_weights and not missing, (
    f"Release is incomplete. weights present: {has_weights}; missing: {missing}. "
    f"Files found: {sorted(os.listdir(MODEL_DIR))}")
print("release OK:", sorted(os.listdir(MODEL_DIR)))

## 3. Load the model + tokenizer

The saved release already has the resized embeddings and LM-head bias baked in, so this is a
plain `from_pretrained` — no manual patching needed. Generation defaults (`|` token, decoding
mode, temperature) are read from the training run's `config.json.bak` so they match how the model
was trained.

In [ ]:
# === Load tokenizer + model ===================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = EsmForMaskedLM.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

# special-token ids and the 20 canonical amino acids (generation is restricted to these)
MASK_ID = tokenizer.mask_token_id
PAD_ID  = tokenizer.pad_token_id
CLS_ID  = tokenizer.cls_token_id
EOS_ID  = tokenizer.eos_token_id

# generation defaults from the training run (with safe fallbacks)
RUN_CFG = {}
_bak = os.path.join(MODEL_DIR, "config.json.bak")
if os.path.exists(_bak):
    with open(_bak) as f: RUN_CFG = json.load(f)
PIPE_TOKEN          = RUN_CFG.get("pipe_token", "|")
DEFAULT_DECODING    = RUN_CFG.get("gen_decoding", "min_entropy")
DEFAULT_TEMPERATURE = RUN_CFG.get("gen_temperature", 1.0)
DEFAULT_UNMASK      = RUN_CFG.get("gen_unmask_per_step", 1)
PIPE_ID = tokenizer.convert_tokens_to_ids(PIPE_TOKEN)

AA     = list("ACDEFGHIKLMNPQRSTVWY")
AA_IDS = torch.tensor(tokenizer.convert_tokens_to_ids(AA), dtype=torch.long)

# length distribution for sampling generation lengths
with open(os.path.join(MODEL_DIR, "length_pairs.json")) as f:
    LENGTH_PAIRS = json.load(f)   # list of [heavy_len, light_len]

# sanity: the resized vocab/bias must line up, or logits are wrong
V = len(tokenizer)
assert model.config.vocab_size == V == model.lm_head.bias.numel(), (
    f"vocab mismatch: tokenizer {V}, model.config {model.config.vocab_size}, "
    f"bias {model.lm_head.bias.numel()} -- did you load the matching tokenizer?")
print(f"loaded model ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params) | "
      f"vocab {V} | pipe id {PIPE_ID} | {len(LENGTH_PAIRS):,} length pairs")
print(f"defaults: decoding={DEFAULT_DECODING}, T={DEFAULT_TEMPERATURE}, "
      f"unmask_per_step={DEFAULT_UNMASK}")

## 4. Generation functions

This is the sampler (replaces a `generate.py`). `generate()` produces fresh paired antibodies;
`inpaint()` redesigns a residue range while holding the rest of a template fixed. Both use the
faithful sampler: minimum-entropy position selection uses the model's native (T=1) distribution,
and temperature is applied only to token sampling.

In [ ]:
@torch.no_grad()
def generate(model, n=16, decoding=None, temperature=None, unmask_per_step=None, seed=None):
    """Unconditional generation. Returns a list of {"heavy", "light"} dicts."""
    decoding        = DEFAULT_DECODING    if decoding        is None else decoding
    temperature     = DEFAULT_TEMPERATURE if temperature     is None else temperature
    unmask_per_step = DEFAULT_UNMASK      if unmask_per_step  is None else unmask_per_step
    if seed is not None: set_seed(seed)
    model.eval()

    # sample (heavy_len, light_len) pairs from the empirical training distribution
    chosen = [random.choice(LENGTH_PAIRS) for _ in range(n)]
    rows = []
    for h_len, l_len in chosen:
        ids = [CLS_ID] + [MASK_ID]*h_len + [PIPE_ID] + [MASK_ID]*l_len + [EOS_ID]
        rows.append(ids)
    maxlen = max(len(r) for r in rows)
    input_ids = torch.full((n, maxlen), PAD_ID, dtype=torch.long)
    attn = torch.zeros((n, maxlen), dtype=torch.long)
    for i, r in enumerate(rows):
        input_ids[i, :len(r)] = torch.tensor(r); attn[i, :len(r)] = 1
    input_ids, attn = input_ids.to(DEVICE), attn.to(DEVICE)
    is_masked = (input_ids == MASK_ID)

    aa_ids = AA_IDS.to(DEVICE)
    disallowed = torch.ones(model.config.vocab_size, dtype=torch.bool, device=DEVICE)
    disallowed[aa_ids] = False  # only canonical AAs allowed at generated positions

    steps = int(is_masked.sum(dim=1).max().item())
    for _ in range(steps):
        if not is_masked.any(): break
        raw = model(input_ids=input_ids, attention_mask=attn).logits   # NO temperature here
        raw[:, :, disallowed] = float("-inf")
        base = torch.softmax(raw, dim=-1)                              # T=1 -> position selection
        ent  = -(base * torch.log(base + 1e-12)).sum(dim=-1)
        ent  = ent.masked_fill(~is_masked, float("inf"))
        sample_probs = torch.softmax(raw / max(temperature, 1e-6), dim=-1)  # T -> token sampling

        for i in range(n):
            mrow = is_masked[i]
            if not mrow.any(): continue
            k = min(unmask_per_step, int(mrow.sum().item()))
            if decoding == "min_entropy":
                pos = torch.topk(ent[i], k, largest=False).indices
            else:  # random order
                cand = torch.nonzero(mrow, as_tuple=False).squeeze(-1)
                pos = cand[torch.randperm(cand.numel(), device=DEVICE)[:k]]
            for p in pos.tolist():
                tok = torch.multinomial(sample_probs[i, p], 1).item()
                input_ids[i, p] = tok
                is_masked[i, p] = False

    out = []
    for i in range(n):
        toks = input_ids[i][attn[i].bool()].tolist()
        s = tokenizer.decode(toks, skip_special_tokens=False)
        s = s.replace("<cls>", "").replace("<eos>", "").replace("<pad>", "").replace(" ", "")
        if PIPE_TOKEN in s:
            h, l = s.split(PIPE_TOKEN, 1)
        else:
            h, l = s, ""
        out.append({"heavy": h, "light": l})
    return out


@torch.no_grad()
def inpaint(model, heavy, light, mask_spans, temperature=None, decoding=None, unmask_per_step=None):
    """Redesign residue ranges on a template. mask_spans: list of ('H'|'L', start, end),
    0-based half-open. Returns (new_heavy, new_light)."""
    decoding        = DEFAULT_DECODING    if decoding        is None else decoding
    temperature     = DEFAULT_TEMPERATURE if temperature     is None else temperature
    unmask_per_step = DEFAULT_UNMASK      if unmask_per_step  is None else unmask_per_step
    model.eval()
    h = list(heavy); l = list(light)
    seq_ids = [CLS_ID] + tokenizer.convert_tokens_to_ids(h) + [PIPE_ID] \
              + tokenizer.convert_tokens_to_ids(l) + [EOS_ID]
    seq_ids = torch.tensor(seq_ids, dtype=torch.long)
    h_off, l_off = 1, 1 + len(h) + 1
    for chain, st, en in mask_spans:
        base = h_off if chain == "H" else l_off
        seq_ids[base + st: base + en] = MASK_ID
    input_ids = seq_ids.unsqueeze(0).to(DEVICE)
    attn = torch.ones_like(input_ids)
    is_masked = (input_ids == MASK_ID)
    disallowed = torch.ones(model.config.vocab_size, dtype=torch.bool, device=DEVICE)
    disallowed[AA_IDS.to(DEVICE)] = False
    for _ in range(int(is_masked.sum())):
        if not is_masked.any(): break
        raw = model(input_ids=input_ids, attention_mask=attn).logits
        raw[:, :, disallowed] = float("-inf")
        base = torch.softmax(raw, -1)
        ent = -(base * torch.log(base + 1e-12)).sum(-1).masked_fill(~is_masked, float("inf"))
        sample_probs = torch.softmax(raw / max(temperature, 1e-6), -1)
        k = min(unmask_per_step, int(is_masked.sum()))
        pos = (torch.topk(ent[0], k, largest=False).indices if decoding == "min_entropy"
               else torch.nonzero(is_masked[0]).squeeze(-1)[:k])
        for p in pos.tolist():
            input_ids[0, p] = torch.multinomial(sample_probs[0, p], 1).item(); is_masked[0, p] = False
    toks = input_ids[0].tolist()
    s = tokenizer.decode(toks, skip_special_tokens=False)
    s = s.replace("<cls>", "").replace("<eos>", "").replace("<pad>", "").replace(" ", "")
    nh, nl = s.split(PIPE_TOKEN, 1)
    return nh, nl

print("generate() and inpaint() ready")

## 5. Generate antibodies

`generate(model, n=...)` returns a list of `{"heavy", "light"}` dicts. Pass `seed=` for
reproducibility, `temperature=` to trade naturalness for diversity (≈1.0 is the trained default),
`decoding="random"` to use random-order instead of minimum-entropy.

In [ ]:
samples = generate(model, n=8, seed=1)
for i, g in enumerate(samples):
    print(f"[{i}] H({len(g['heavy'])}): {g['heavy']}")
    print(f"    L({len(g['light'])}): {g['light']}")

## 6. Save candidates to CSV

A simple way to hand generated sequences to downstream tools.

In [ ]:
import pandas as pd
batch = generate(model, n=64, seed=123)
df = pd.DataFrame(batch)
df["heavy_len"] = df["heavy"].str.len()
df["light_len"] = df["light"].str.len()
out_csv = "generated_antibodies.csv"
df.to_csv(out_csv, index=False)
print(f"wrote {len(df)} sequences -> {out_csv}")
df.head()

## 7. Redesign a template (inpainting)

Keep a template fixed and mask the range you want to redesign. Spans are 0-based half-open on the
chosen chain. (For real HCDR3 redesign, get the bounds from ANARCI/IMGT numbering.) Here we
generate a template, then redesign a stretch of its heavy chain.

In [ ]:
tpl = generate(model, n=1, seed=7)[0]
new_h, new_l = inpaint(model, tpl["heavy"], tpl["light"], [("H", 25, 35)], seed=None)
print("template H:", tpl["heavy"])
print("redesign H:", new_h)
print("(light unchanged):", new_l == tpl["light"])

## 8. Quick sanity checks (optional)

Uniqueness and validity need no reference data. For the paper's **naturalness** scores (AbLang2 /
p-IgGen), install those packages (commented in the deps cell) and score the `heavy`/`light` pairs;
they're left out here because the packages are heavy.

In [ ]:
def pct_unique(items):
    keys = [g["heavy"] + "|" + g["light"] for g in items]
    return len(set(keys)) / max(len(keys), 1)

def validity(items):
    ok = sum(1 for g in items if g["heavy"] and g["light"]
             and set(g["heavy"]) <= set(AA) and set(g["light"]) <= set(AA))
    return ok / max(len(items), 1)

print(f"% unique : {pct_unique(batch):.3f}")
print(f"% valid  : {validity(batch):.3f}")

# example temperature sweep (inference only -- no retraining needed)
for T in (0.5, 1.0, 1.5):
    s = generate(model, n=16, temperature=T, seed=0)
    print(f"T={T}: unique={pct_unique(s):.2f} valid={validity(s):.2f}")

## Troubleshooting

- **`vocab mismatch` assertion** — you loaded a stock ESM-2 tokenizer instead of the one in the
  release. Always load the tokenizer from `MODEL_DIR`; it carries the added `|` token.
- **Garbage / repetitive output** — usually a tokenizer/version mismatch. Pin `transformers` to the
  version in the README's run-provenance section.
- **Slow generation** — each sequence is decoded one token per forward pass (faithful). Raise
  `unmask_per_step` (e.g. 4) to trade a little fidelity for speed.
- **No GPU** — it runs on CPU, just slowly.

### Does this replace `requirements.txt` and `generate.py`?
Yes. The install cell (Section 1) is the dependency spec, and Sections 3–4 are the model load and
sampler. This single notebook is all the team needs to use the model. A separate `generate.py` is
only worth adding if someone wants to import the functions into a non-notebook script.
